# 06 — AI-Assisted Feature Enrichment

## Objective

This notebook explores how Large Language Models (LLMs)
can transform unstructured Steam game descriptions into
structured features.

Using the 50-game sample collected in September 2026,
the objective is to:

1. Define a consistent classification framework.
2. Extract qualitative features using an LLM.
3. Validate the AI-generated classifications.
4. Explore their potential value for market segmentation.

The resulting features will complement the historical
market analysis and support the interpretation of
selected games.

In [1]:
from pathlib import Path
import pandas as pd
import json

# Define project paths
project_root = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

data_path = (
    project_root
    / "data/processed/steam_current_enriched_2026.csv"
)

# Load the enriched sample
df_ai = pd.read_csv(data_path)

print("Dataset shape:", df_ai.shape)

print(
    "Available descriptions:",
    df_ai["short_description"].notna().sum()
)

print(
    "Historical overlap:",
    df_ai["present_in_march2025"].sum()
)

display(
    df_ai[
        ["appid", "name", "short_description"]
    ].head()
)

Dataset shape: (50, 17)
Available descriptions: 50
Historical overlap: 25


,appid,name,short_description
0,230410,Warframe,Awaken as an unstoppable warrior and battle al...
1,3669870,CONTROL Resonant,Explore a warped Manhattan on the brink of par...
2,1867240,WARDOGS,WARDOGS is a TACTICAL ALL OUT WARFARE FPS wher...
3,730,Counter-Strike 2,"For over two decades, Counter-Strike has offer..."
4,1142710,Total War: WARHAMMER III,The cataclysmic conclusion to the Total War: W...


## 1. Text Data Audit

Before applying AI-assisted feature extraction, we need
to assess the quality of the collected game descriptions.

Since the objective is to use an LLM, preprocessing
will focus on removing formatting issues while
preserving the original meaning and context.

In [2]:
# Audit text quality before preprocessing

descriptions = df_ai["short_description"]

print("Total descriptions:", len(descriptions))

print("Missing descriptions:", descriptions.isna().sum())

print(
    "Empty descriptions:",
    descriptions.fillna("").str.strip().eq("").sum()
)

print(
    "Descriptions containing HTML tags:",
    descriptions.fillna("").str.contains(
        r"<[^>]+>",
        regex=True
    ).sum()
)

print(
    "Duplicate descriptions:",
    descriptions.duplicated().sum()
)

print("\nDescription lengths:")
display(descriptions.str.len().describe())

# Inspect a few complete descriptions
for row in df_ai[
    ["name", "short_description"]
].head(3).itertuples(index=False):

    print(f"\nGAME: {row.name}")
    print(f"DESCRIPTION: {row.short_description}")
    print("-" * 60)

Total descriptions: 50
Missing descriptions: 0
Empty descriptions: 0
Descriptions containing HTML tags: 0
Duplicate descriptions: 0

Description lengths:


count     50.000000
mean     214.300000
std       59.029999
min      103.000000
25%      170.500000
50%      218.500000
75%      267.000000
max      300.000000
Name: short_description, dtype: float64


GAME: Warframe
DESCRIPTION: Awaken as an unstoppable warrior and battle alongside your friends in this story-driven free-to-play online action game
------------------------------------------------------------

GAME: CONTROL Resonant
DESCRIPTION: Explore a warped Manhattan on the brink of paranatural annihilation in this thrilling action-adventure RPG. Unleash the extraordinary powers of Dylan Faden as he fights for his humanity and our survival against a reality-bending cosmic threat.
------------------------------------------------------------

GAME: WARDOGS
DESCRIPTION: WARDOGS is a TACTICAL ALL OUT WARFARE FPS where one hundred players split across three teams fight over the control zone. Set in the derelict industrial mountains of Eastern Europe on a destructible battlefield where teamwork pays and cash is king.
------------------------------------------------------------


## 2. Text Preprocessing

The initial data audit identified no missing descriptions,
duplicate descriptions or HTML tags.

Because this project uses a Large Language Model,
preprocessing is intentionally limited.

The objective is to remove unnecessary formatting while
preserving the semantic context needed for classification.

In [3]:
import html

# Preserve the original descriptions
df_ai["description_clean"] = (
    df_ai["short_description"]
    .apply(html.unescape)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Verify the preprocessing
print(
    "Descriptions modified:",
    (
        df_ai["description_clean"]
        != df_ai["short_description"]
    ).sum()
)

print(
    "Descriptions available:",
    df_ai["description_clean"].notna().sum()
)

display(
    df_ai[
        ["name", "description_clean"]
    ].head(3)
)

Descriptions modified: 1
Descriptions available: 50


,name,description_clean
0,Warframe,Awaken as an unstoppable warrior and battle al...
1,CONTROL Resonant,Explore a warped Manhattan on the brink of par...
2,WARDOGS,WARDOGS is a TACTICAL ALL OUT WARFARE FPS wher...


## 3. AI-Assisted Classification

A Large Language Model (LLM) will be used to extract
structured game-positioning features from the collected
Steam descriptions.

The workflow uses prompt engineering, structured outputs
and human evaluation.

Classification will be performed using ChatGPT,
and the resulting data will be integrated and analyzed
using Python.

In [4]:
# Export descriptions for AI-assisted classification

ai_input = df_ai[
    ["appid", "name", "description_clean"]
].copy()

output_path = (
    project_root
    / "data/processed/steam_ai_input_2026.csv"
)

ai_input.to_csv(
    output_path,
    index=False
)

print("AI input file created successfully!")
print("Games exported:", len(ai_input))

AI input file created successfully!
Games exported: 50


## 4. AI Classification Integration

The AI-generated classifications are imported and
matched with the original game dataset using Steam App IDs.

Before analyzing the results, we verify data completeness,
identifier uniqueness and successful integration.

These classifications are preliminary and require
human validation.

In [5]:
# Import AI-generated classifications

draft_path = (
    project_root
    / "data/processed/steam_ai_classifications_2026_draft.csv"
)

df_labels = pd.read_csv(draft_path)

# Verify identifiers and completeness
print("AI classifications:", len(df_labels))
print("Duplicate App IDs:", df_labels["appid"].duplicated().sum())

print(
    "All game IDs match:",
    set(df_ai["appid"]) == set(df_labels["appid"])
)

# Merge classifications with original data
df_merged = df_ai.merge(
    df_labels.drop(columns=["name"]),
    on="appid",
    how="left",
    validate="one_to_one",
    indicator=True
)

print("\nMerge results:")
print(df_merged["_merge"].value_counts())

print(
    "\nMissing primary classifications:",
    df_merged["experience_primary"].isna().sum()
)

# Display the integrated classifications
display(
    df_merged[
        [
            "name",
            "experience_primary",
            "experience_secondary",
            "social_orientation"
        ]
    ].head(10)
)

AI classifications: 50
Duplicate App IDs: 0
All game IDs match: True

Merge results:
_merge
both          50
left_only      0
right_only     0
Name: count, dtype: int64

Missing primary classifications: 0


,name,experience_primary,experience_secondary,social_orientation
0,Warframe,Narrative,Action/Combat,Cooperative
1,CONTROL Resonant,Exploration,Narrative,Unspecified
2,WARDOGS,Competitive,Strategy,Competitive multiplayer
3,Counter-Strike 2,Competitive,NaN,Multiplayer unspecified
4,Total War: WARHAMMER III,Strategy,Action/Combat,Unspecified
5,SILENT HILL: Townfall,Narrative,Action/Combat,Unspecified
6,Apex Legends™,Competitive,Strategy,Competitive multiplayer
7,Disney Dreamlight Valley,Simulation,Narrative,Unspecified
8,Marvel Rivals,Competitive,Strategy,Competitive multiplayer
9,HELLDIVERS™ 2,Action/Combat,NaN,Unspecified


### Human Evaluation Results

A manual evaluation was conducted on a sample of
10 games to assess the consistency of AI-generated
classifications.

The evaluation produced the following results:

- Primary experience agreement: 50% (5/10).
- Social orientation agreement: 60% (6/10).

The results highlight the difficulty of identifying
a single dominant experience when game descriptions
mention multiple gameplay characteristics.

Some disagreements also reflect differences in
classification specificity rather than completely
contradictory interpretations.

These findings demonstrate the importance of
clear classification guidelines and human review
when using LLM-generated features.

This small exploratory evaluation does not provide
a definitive measure of model accuracy.

## 5. Human Evaluation and Classification Analysis

A sample of 10 games was independently classified
to evaluate agreement with the AI-generated labels.

The evaluation focuses on primary gameplay experience
and social orientation.

The full set of AI classifications is then explored
to identify the distribution of the generated features.

In [6]:
# Load corrected human evaluation results

processed_path = project_root / "data/processed"

df_human = pd.read_csv(
    processed_path / "steam_ai_human_review_corrected.csv"
)

df_comparison = pd.read_csv(
    processed_path / "steam_ai_human_vs_ai_corrected.csv"
)

# Verify evaluation results
print("Games reviewed:", len(df_human))

print(
    "Primary experience agreement:",
    df_comparison["primary_agreement"].sum(),
    "/", len(df_comparison)
)

print(
    "Social orientation agreement:",
    df_comparison["social_agreement"].sum(),
    "/", len(df_comparison)
)

# Analyze AI-generated classifications
print("\nPRIMARY EXPERIENCE DISTRIBUTION")
display(
    df_merged["experience_primary"].value_counts()
)

print("\nSOCIAL ORIENTATION DISTRIBUTION")
display(
    df_merged["social_orientation"].value_counts()
)

Games reviewed: 10
Primary experience agreement: 5 / 10
Social orientation agreement: 6 / 10

PRIMARY EXPERIENCE DISTRIBUTION


experience_primary
Action/Combat        9
Competitive          8
Simulation           7
Narrative            6
Strategy             6
Exploration          5
Survival/Crafting    4
Unclear              3
Other                1
Social/Creative      1
Name: count, dtype: int64


SOCIAL ORIENTATION DISTRIBUTION


social_orientation
Unspecified                25
Multiplayer unspecified     9
Competitive multiplayer     7
Cooperative                 6
Mixed                       3
Name: count, dtype: int64

## 6. Results and Final Conclusion

### AI Classification Results

AI-assisted feature extraction was performed on
50 Steam game descriptions.

The most frequently assigned primary experiences were:

- Action/Combat: 18%
- Competitive: 16%
- Simulation: 14%
- Narrative: 12%
- Strategy: 12%

Social orientation could not be determined for
25 games (50%), highlighting the limitations
of the available textual information.

### Human Evaluation

An independent manual review was conducted
on a sample of 10 games.

The evaluation produced:
- 50% agreement on primary experience.
- 60% agreement on social orientation.

These results highlight the subjective nature
of identifying dominant gameplay characteristics
and the importance of human validation.

The small evaluation sample does not provide
a definitive measure of model accuracy.

### Conclusion

This experiment demonstrates how prompt engineering
and Large Language Models can transform unstructured
game descriptions into structured features.

However, the generated classifications remain
exploratory and should not be treated as
verified ground truth.

The enriched dataset may support the interpretation
of selected games within the broader Steam
market analysis.

The main machine learning segmentation will
continue to rely on the larger historical dataset.

In [7]:
# Save the final AI-enriched dataset

final_ai_path = (
    project_root
    / "data/processed/steam_ai_enriched_2026.csv"
)

# Remove the temporary merge indicator
df_final_ai = df_merged.drop(
    columns=["_merge"],
    errors="ignore"
).copy()

# Export the enriched dataset
df_final_ai.to_csv(
    final_ai_path,
    index=False
)

# Final checks
print("Final dataset saved successfully!")
print("Shape:", df_final_ai.shape)
print("Unique games:", df_final_ai["appid"].nunique())

print(
    "Missing primary classifications:",
    df_final_ai["experience_primary"].isna().sum()
)

print("\nSaved to:")
print(final_ai_path)

Final dataset saved successfully!
Shape: (50, 25)
Unique games: 50
Missing primary classifications: 0

Saved to:
/Users/AurianeB/inside-steam-analytics/data/processed/steam_ai_enriched_2026.csv
